## 18. Эксперимент без RD-фичей ("как будто есть только Bybit")

**Цель:** на тех же данных, сплите, моделях и бэктесте, что в 13/15/16 ноутбуках, проверить качество моделей,
если полностью убрать все фичи, связанные с `rd_value` и разметкой заказчика (включая `rd_regime`, `rd_regime_transition`).

То есть оставляем только то, что можно получить от биржи Bybit: OHLCV + time (и производные от них).

Дальше сравниваем с результатами LightGBM + seq_5_15_30_60 из 13/15/16, чтобы оценить вклад RD-данных.

In [1]:
import sys, os, numpy as np, pandas as pd, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == '05_experiments' else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

HAS_XGB, HAS_LGB, HAS_CAT = False, False, False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    pass
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    pass
try:
    import catboost as cb
    HAS_CAT = True
except ImportError:
    pass

COMMISSION_RT = 0.001
THRESHOLD = 0.75  # band 25-75
TARGET_COL = 'target'

print('Root:', _root)
print('XGB:', HAS_XGB, '| LGB:', HAS_LGB, '| CAT:', HAS_CAT)

Root: c:\project\trading_bot_2Engine
XGB: True | LGB: True | CAT: True


### 1. Загрузка данных и базовые фичи (как в 13/16)

In [2]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

valid = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'

train_dates = set(dates[:8])
val_dates   = set([dates[8]])
test_dates  = set([dates[9]])

print('Dates: train=', f'{min(train_dates)}..{max(train_dates)}', '| val=', dates[8], '| test=', dates[9])
print('Rows total:', len(valid))
print('BASE_FEATURES (22):', BASE_FEATURES)

Dates: train= 2026-02-01..2026-02-08 | val= 2026-02-09 | test= 2026-02-10
Rows total: 186063
BASE_FEATURES (22): ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'rd_regime', 'rd_regime_transition']


### 2. Отбрасываем все RD-фичи (оставляем только то, что можно получить от биржи)

In [3]:
# RD-фичи: всё, что строится напрямую из rd_value или его режимов
def is_rd_feature(c: str) -> bool:
    if c.startswith('rd_'):
        return True
    if c == 'abs_rd':
        return True
    return False

BASE_NO_RD = [c for c in BASE_FEATURES if not is_rd_feature(c)]
print('BASE_NO_RD (без RD):', len(BASE_NO_RD), BASE_NO_RD)

BASE_NO_RD (без RD): 13 ['ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']


### 3. Rolling-фичи без RD (seq_5_15_30_60 только по OHLCV/time)

Берём под rolling только фичи, которые можно посчитать из OHLCV + времени, **без** rd_value и rd_regime:

In [4]:
SEQ_WINDOWS = [5, 15, 30, 60]
KEY_FEATS_NO_RD = [c for c in BASE_NO_RD if c in ['ret_1', 'ret_5', 'rsi_14', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position']]
print('KEY_FEATS_NO_RD:', KEY_FEATS_NO_RD)

grp = valid.groupby('session_key', group_keys=False)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS_NO_RD:
        valid[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
        valid[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_NO_RD = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS_NO_RD for s in ('mean', 'std')]
FEATURES_BASE_NO_RD = BASE_NO_RD[:]
FEATURES_SEQ_NO_RD = FEATURES_BASE_NO_RD + SEQ_NO_RD

train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df   = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df  = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

print('FEATURES_BASE_NO_RD:', len(FEATURES_BASE_NO_RD))
print('FEATURES_SEQ_NO_RD:', len(FEATURES_SEQ_NO_RD))
print('Rows (train/val/test):', len(train_df), len(val_df), len(test_df))

KEY_FEATS_NO_RD: ['ret_1', 'ret_5', 'rsi_14', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position']
FEATURES_BASE_NO_RD: 13
FEATURES_SEQ_NO_RD: 69
Rows (train/val/test): 146693 24014 15356


### 4. Бэктест (signal_flip) — тот же, что в 13/15/16

In [5]:
def backtest_prod_hold(proba, ret, session_ids, threshold=THRESHOLD, commission_rt=COMMISSION_RT):
    """signal_flip: BUY >= thr, SELL <= 1-thr, HOLD держим позицию."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}

ret_val = val_df['ret_next'].values
sess_val = val_df['session_key'].values
ret_test = test_df['ret_next'].values
sess_test = test_df['session_key'].values

### 5. Обучение моделей на BASE_NO_RD и SEQ_NO_RD

In [6]:
def make_models():
    models = [
        ('logreg', LogisticRegression(max_iter=1200, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42)),
    ]
    if HAS_XGB:
        models.append(('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')))
    if HAS_LGB:
        models.append(('lgb', lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)))
    if HAS_CAT:
        models.append(('cat', cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)))
    return models

def fit_and_eval(name, model, feat_cols, df_tr, df_va, df_te):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(df_tr[feat_cols].fillna(0))
    Xva = scaler.transform(df_va[feat_cols].fillna(0))
    Xte = scaler.transform(df_te[feat_cols].fillna(0))
    ytr, yva, yte = df_tr['y'].values, df_va['y'].values, df_te['y'].values
    model.fit(Xtr, ytr)
    pva = model.predict_proba(Xva)[:, 1]
    pte = model.predict_proba(Xte)[:, 1]
    auc_val = roc_auc_score(yva, pva)
    auc_test = roc_auc_score(yte, pte)
    bt_val = backtest_prod_hold(pva, ret_val, sess_val, threshold=THRESHOLD)
    bt_test = backtest_prod_hold(pte, ret_test, sess_test, threshold=THRESHOLD)
    return {
        'model': name,
        'auc_val': auc_val, 'auc_test': auc_test,
        'net_val_%': bt_val['net_%'], 'trades_val': bt_val['trades'], 'avg_val_%_per_trade': bt_val['avg_%_per_trade'],
        'net_test_%': bt_test['net_%'], 'trades_test': bt_test['trades'], 'avg_test_%_per_trade': bt_test['avg_%_per_trade'],
    }

In [7]:
results = []

for name, model in make_models():
    # baseline без RD
    res_base = fit_and_eval(f'{name}_base_no_rd', model, FEATURES_BASE_NO_RD, train_df, val_df, test_df)
    res_base['feature_set'] = 'base_no_rd'
    results.append(res_base)

    # sequence 5/15/30/60 без RD (rolling только по OHLCV/time-фичам)
    res_seq = fit_and_eval(f'{name}_seq_no_rd', model, FEATURES_SEQ_NO_RD, train_df, val_df, test_df)
    res_seq['feature_set'] = 'seq_5_15_30_60_no_rd'
    results.append(res_seq)

res_df = pd.DataFrame(results)
res_df = res_df[['model', 'feature_set', 'auc_val', 'auc_test', 'net_val_%', 'trades_val', 'avg_val_%_per_trade', 'net_test_%', 'trades_test', 'avg_test_%_per_trade']]
res_df = res_df.sort_values(['feature_set', 'model']).reset_index(drop=True)
res_df

,model,feature_set,auc_val,auc_test,net_val_%,trades_val,avg_val_%_per_trade,net_test_%,trades_test,avg_test_%_per_trade
0,cat_base_no_rd,base_no_rd,0.481308,0.511650,22.679503,18,1.259972,1.377843,1,1.377843
1,lgb_base_no_rd,base_no_rd,0.482184,0.520338,17.293465,32,0.540421,1.524002,8,0.190500
2,logreg_base_no_rd,base_no_rd,0.496603,0.513039,0.000000,0,NaN,0.000000,0,NaN
3,rf_base_no_rd,base_no_rd,0.488802,0.516769,5.594663,6,0.932444,0.000000,0,NaN
4,xgb_base_no_rd,base_no_rd,0.483870,0.516819,90.994982,111,0.819775,-2.292589,15,-0.152839
5,cat_seq_no_rd,seq_5_15_30_60_no_rd,0.758130,0.732777,914.743729,667,1.371430,480.130301,382,1.256886
6,lgb_seq_no_rd,seq_5_15_30_60_no_rd,0.729681,0.712615,556.513083,503,1.106388,332.578553,251,1.325014
7,logreg_seq_no_rd,seq_5_15_30_60_no_rd,0.510837,0.532453,14.169208,1,14.169208,21.340600,2,10.670300
8,rf_seq_no_rd,seq_5_15_30_60_no_rd,0.681886,0.661826,-0.527469,17,-0.031028,27.882120,9,3.098013
9,xgb_seq_no_rd,seq_5_15_30_60_no_rd,0.734853,0.714592,748.137241,652,1.147450,448.325241,331,1.354457


### 6. Интерпретация: вклад RD-данных

Чтобы оценить вклад RD, нужно сравнить эту таблицу с результатами из ноутбуков 13/15/16 для конфигураций с RD-фичами
(особенно LightGBM + seq_5_15_30_60). На что смотрим:

- падение AUC (val/test) без RD;
- падение `avg_%_per_trade` на test-дне;
- изменение количества сделок.

Если без RD и baseline, и sequence дают заметно более слабый AUC и avg/trade,
значит RD-данные заказчика действительно несут ценную информацию, а не просто дублируют OHLCV/time.

### 7. Конкретные выводы по результатам (без RD)

1. **Baseline без RD (13 фичей)** практически не даёт сигналов:
   - AUC val/test ~ 0.48–0.52 (около случайного угадывания).
   - `net_%` и `avg_%_per_trade` на test-дне около нуля или шумовые значения при очень малом числе сделок.
   - Вывод: одни только OHLCV/time-фичи без sequence и без RD **почти не несут торговой альфы**.

2. **Sequence 5/15/30/60 без RD (69 фичей)** уже даёт ощутимую информативность:
   - Лучшая модель по AUC и PnL — CatBoost (`cat_seq_no_rd`), далее XGBoost (`xgb_seq_no_rd`).
   - `cat_seq_no_rd`:
     - AUC val ≈ 0.76, test ≈ 0.73.
     - test: `net_% ≈ +480%`, `trades ≈ 380`, `avg_%_per_trade ≈ 1.26%`.
   - `xgb_seq_no_rd`:
     - AUC val ≈ 0.73, test ≈ 0.71.
     - test: `net_% ≈ +448%`, `trades ≈ 330`, `avg_%_per_trade ≈ 1.35%`.
   - Вывод: даже без RD данные биржи (OHLCV + time) в sequence-формате уже позволяют строить **неплохие торговые модели**.

3. **Сравнение с конфигурацией *с RD* (LightGBM + seq_5_15_30_60 из 16-го шага):
   - Там AUC val ≈ 0.90+, test ≈ 0.80+ — это **существенно выше**, чем 0.71–0.73 без RD.
   - `avg_%_per_trade` на test-дне сопоставим или чуть выше: около 1.3–1.4%.
   - Количество сделок и net_% в RD-конфигурации также выше, при том же уровне комиссии и логике сделок.
   - Вывод: **RD-блок заметно усиливает разделяющую способность модели (AUC), сохраняя высокий avg/trade**.

4. **Итог по вкладу RD:**
   - Без RD: sequence-фичи по OHLCV/time уже дают рабочую стратегию (особенно CatBoost/XGBoost), но на уровне AUC ≈ 0.7+.
   - С RD: LightGBM на тех же окнах 5/15/30/60 выходит на AUC ≈ 0.90/0.80 и более высокий суммарный PnL.
   - Следовательно, RD-данные заказчика **несут существенную дополнительную информацию**, особенно для улучшения качества классификации и устойчивости стратегии, а не просто дублируют сигналы классических ценовых индикаторов.